# South Africa — Rival framings model notebook

This notebook is meant to run **after** the main South Africa project notebook. It focuses only on the extra analysis needed for the results chapter and debate:

1. Compare the **South Africa justice framing** with rival framings.
2. Load or run **Prioritarian** and **Utilitarian** Pareto fronts.
3. Re-evaluate selected policies to compute South Africa-specific burden indicators.
4. Produce tables for Chapter 4: policy performance, South Africa burden, robustness, and rival welfare comparison.

The key idea is: not every rival framing needs a completely separate optimisation. The welfare framings are run through different welfare functions; the other framings are mostly evaluated by post-processing the same policy space.


## 0. Setup and file locations

This cell follows the same path logic as the main South Africa notebook. It assumes the folder structure contains:

- `JUSTICE-main/`
- `config/`
- `A- Project G15/results/`
- `A- Project G15/plots/`


In [ ]:
import os
import sys
import json
import glob
import re
import shlex
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    _NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path.cwd().resolve()

# Find repo root: the folder that contains JUSTICE-main and config
_REPO_ROOT = _NOTEBOOK_DIR
for candidate in [_NOTEBOOK_DIR] + list(_NOTEBOOK_DIR.parents):
    if (candidate / "JUSTICE-main").exists() and (candidate / "config").exists():
        _REPO_ROOT = candidate
        break

_JUSTICE_ROOT = (_REPO_ROOT / "JUSTICE-main").resolve()
_CONFIG_DIR = (_REPO_ROOT / "config").resolve()
_PROJECT_DIR = (_REPO_ROOT / "A- Project G15").resolve()

RESULTS_DIR = (_PROJECT_DIR / "results").resolve()
PLOTS_DIR = (_PROJECT_DIR / "plots").resolve()
TABLES_DIR = (_PROJECT_DIR / "tables").resolve()

for path in [RESULTS_DIR, PLOTS_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(_JUSTICE_ROOT) not in sys.path:
    sys.path.insert(0, str(_JUSTICE_ROOT))

# Some JUSTICE imports assume the working directory is JUSTICE-main.
os.chdir(_JUSTICE_ROOT)

from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction, Economy, DamageFunction, Abatement
from justice.util.data_loader import DataLoader
from justice.util.model_time import TimeHorizon

print("Notebook dir:", _NOTEBOOK_DIR)
print("Repo root:", _REPO_ROOT)
print("Project dir:", _PROJECT_DIR)
print("JUSTICE root:", _JUSTICE_ROOT)
print("Config dir:", _CONFIG_DIR)
print("Results dir:", RESULTS_DIR)
print("Plots dir:", PLOTS_DIR)
print("Tables dir:", TABLES_DIR)


## 1. Define rival framings

These are the framings used in Chapter 4. Only the first two are direct welfare-function runs. The other two are post-processing perspectives.


In [ ]:
framings = pd.DataFrame([
    {
        "Framing": "South Africa justice framing",
        "Model lens": "Prioritarian welfare function + focus on zaf",
        "Main indicator": "abatement_cost_zaf / gross_economic_output_zaf",
        "Political interpretation": "Fairness as proportional burden; ambition is acceptable only if the transition burden is manageable and financed.",
        "Requires new optimisation?": "Yes: PRIORITARIAN run"
    },
    {
        "Framing": "Global efficiency framing",
        "Model lens": "Utilitarian welfare function",
        "Main indicator": "welfare, fraction_above_threshold, aggregate losses",
        "Political interpretation": "Fairness as total global performance; can hide regional burden differences.",
        "Requires new optimisation?": "Yes: UTILITARIAN run, if compute time allows"
    },
    {
        "Framing": "Equal obligation framing",
        "Model lens": "Comparison of mitigation effort across regions",
        "Main indicator": "emission_control_rate_zaf vs developed regions + relative burden",
        "Political interpretation": "Equal mitigation rates can still create unequal economic burdens.",
        "Requires new optimisation?": "No: post-process selected policies"
    },
    {
        "Framing": "High-ambition or survival framing",
        "Model lens": "Climate-risk focused evaluation",
        "Main indicator": "fraction_above_threshold, global_temperature, climate damages",
        "Political interpretation": "Urgency matters, but ambition must be financed to be implementable.",
        "Requires new optimisation?": "No: select low climate-risk policies"
    },
])

framings


In [ ]:
# Save the framing table for the report
framings_csv = TABLES_DIR / "rival_framings_model_operationalisation.csv"
framings_tex = TABLES_DIR / "rival_framings_model_operationalisation.tex"

framings.to_csv(framings_csv, index=False)
framings.to_latex(
    framings_tex,
    index=False,
    escape=True,
    caption="Operationalisation of rival framings in the model results",
    label="tab:rival_framings_model",
)

print("Saved:", framings_csv)
print("Saved:", framings_tex)


## 2. Load configuration and region list

Use the same configuration file as the main notebook. If your main notebook uses a different config path, change `CONFIG_PATH` below.


In [ ]:
# Try common config names. Change this manually if your config has a different name.
candidate_configs = [
    _PROJECT_DIR / "config_student.json",
    _CONFIG_DIR / "config_student.json",
    _CONFIG_DIR / "normative_uncertainty_optimization.json",
    _CONFIG_DIR / "config.json",
]

CONFIG_PATH = next((p for p in candidate_configs if p.exists()), None)
if CONFIG_PATH is None:
    raise FileNotFoundError(
        "No config file found. Set CONFIG_PATH manually to the config used in the main notebook."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    cfg = json.load(fh)

START_YEAR = cfg.get("start_year", 2015)
END_YEAR = cfg.get("end_year", 2300)
TIMESTEP = cfg.get("timestep", 1)
DATA_TIMESTEP = cfg.get("data_timestep", 5)
REFERENCE_SCENARIO = cfg.get("reference_ssp_rcp_scenario_index", cfg.get("reference_index", 2))
N_INPUTS_RBF = cfg.get("n_inputs", 2)
N_RBFS = cfg.get("n_rbfs", N_INPUTS_RBF + 2)
TEMPERATURE_YEAR_OF_INTEREST = cfg.get("temperature_year_of_interest", 2100)
EMISSION_CONTROL_START_YEAR = cfg.get("emission_control_start_year", 2025)

_time_horizon = TimeHorizon(
    start_year=START_YEAR,
    end_year=END_YEAR,
    data_timestep=DATA_TIMESTEP,
    timestep=TIMESTEP,
)

REGION_LIST = list(DataLoader().REGION_LIST)
N_REGIONS = len(REGION_LIST)
N_TIMESTEPS = len(_time_horizon.model_time_horizon)
ZAF_IDX = REGION_LIST.index("zaf")
EC_START_TS = _time_horizon.year_to_timestep(EMISSION_CONTROL_START_YEAR, timestep=TIMESTEP)
TEMP_YEAR_IDX = _time_horizon.year_to_timestep(TEMPERATURE_YEAR_OF_INTEREST, timestep=TIMESTEP)

print("Config path:", CONFIG_PATH)
print("Reference scenario:", REFERENCE_SCENARIO)
print("N regions:", N_REGIONS)
print("N timesteps:", N_TIMESTEPS)
print("South Africa index:", ZAF_IDX)
print("Temperature threshold year index:", TEMP_YEAR_IDX)


## 3. Optional: run the two welfare framings

Only run this if you still need Pareto fronts. If you already have `pareto_front_*.csv` files for `PRIORITARIAN` and `UTILITARIAN`, skip this section and go to Section 4.

Recommended logic:

- **Prioritarian** = South Africa's own justice framing.
- **Utilitarian** = rival global efficiency framing.

Set `RUN_OPTIMISATION = True` only when you are ready to run. Start with a small `NFE_TEST`, then increase.


In [ ]:
RUN_OPTIMISATION = False  # Change to True only when you want to run the optimiser.
NFE_TEST = 500             # Increase later, e.g. 10000, 50000, 100000.
SEEDS = [1]                # Increase later, e.g. [1, 2, 3, 4, 5].
POPULATION_SIZE = 100

# Check welfare-function mapping before running.
for i in range(8):
    try:
        print(i, WelfareFunction.from_index(i))
    except Exception as err:
        print(i, "not available", err)


In [ ]:
if RUN_OPTIMISATION:
    from run_optimization import run_optimization_adaptive
    from justice.util.enumerations import Optimizer, Evaluator
    from ema_workbench import ema_logging

    ema_logging.log_to_stderr(ema_logging.INFO)

    # Usually: 0 = UTILITARIAN, 1 = PRIORITARIAN. The previous cell prints the mapping.
    WELFARE_RUNS = {
        "UTILITARIAN": 0,
        "PRIORITARIAN": 1,
    }

    for welfare_name, swf_index in WELFARE_RUNS.items():
        for seed in SEEDS:
            print(f"Running {welfare_name}, seed={seed}, nfe={NFE_TEST}")
            run_optimization_adaptive(
                config_path=str(CONFIG_PATH),
                nfe=NFE_TEST,
                population_size=POPULATION_SIZE,
                swf=swf_index,
                seed=seed,
                datapath=str(RESULTS_DIR),
                optimizer=Optimizer.EpsNSGAII,
                evaluator=Evaluator.SequentialEvaluator,
                reference_ssp_rcp_scenario_index=REFERENCE_SCENARIO,
            )
else:
    print("RUN_OPTIMISATION is False. Skipping optimisation.")


## 4. Load Pareto fronts

This reads all `pareto_front_*.csv` files under the results folder and keeps metadata from the folder name. The expected folder naming convention is something like:

```text
PRIORITARIAN_100000_1/pareto_front_1.csv
UTILITARIAN_100000_1/pareto_front_1.csv
```


In [ ]:
OBJECTIVE_COLS = [
    "welfare",
    "fraction_above_threshold",
    "welfare_loss_damage",
    "welfare_loss_abatement",
]

MINIMIZE_COLS = ["welfare", "fraction_above_threshold"]
MAXIMIZE_COLS = ["welfare_loss_damage", "welfare_loss_abatement"]

pareto_files = sorted(Path(RESULTS_DIR).rglob("pareto_front_*.csv"))
print(f"Found {len(pareto_files)} Pareto front file(s) in {RESULTS_DIR}")

if not pareto_files:
    raise FileNotFoundError(
        f"No pareto_front_*.csv files found in {RESULTS_DIR}. "
        "Run the optimiser or check RESULTS_DIR."
    )

all_dfs = []
for file in pareto_files:
    df = pd.read_csv(file)
    folder = file.parent.name
    parts = folder.split("_")

    # More robust parsing: find welfare name and numeric parts from the folder.
    welfare_function = parts[0].upper() if parts else "UNKNOWN"
    numeric_parts = [int(p) for p in parts if p.isdigit()]
    nfe = numeric_parts[0] if len(numeric_parts) >= 1 else np.nan
    seed_folder = numeric_parts[-1] if len(numeric_parts) >= 2 else np.nan

    seed_match = re.search(r"pareto_front_(\d+)\.csv", file.name)
    seed_file = int(seed_match.group(1)) if seed_match else np.nan
    seed = seed_file if not np.isnan(seed_file) else seed_folder

    df["welfare_function"] = welfare_function
    df["nfe"] = nfe
    df["seed"] = seed
    df["source_file"] = str(file)
    all_dfs.append(df)

all_results = pd.concat(all_dfs, ignore_index=True)

missing = [c for c in OBJECTIVE_COLS if c not in all_results.columns]
if missing:
    raise KeyError(f"Missing objective columns: {missing}")

run_summary = (
    all_results
    .groupby(["welfare_function", "nfe", "seed"], dropna=False)
    .size()
    .reset_index(name="n_pareto_solutions")
    .sort_values(["welfare_function", "nfe", "seed"])
)

display(run_summary)
display(all_results[OBJECTIVE_COLS + ["welfare_function"]].groupby("welfare_function").describe().round(4))


## 5. Build reference sets per welfare framing

This merges seeds within each welfare function and filters the set to non-dominated solutions. This avoids relying on one random seed.


In [ ]:
def to_minimization_objectives(df, objective_cols=OBJECTIVE_COLS):
    """Convert all objectives to minimization direction for dominance filtering."""
    vals = df[objective_cols].astype(float).copy()
    for col in MAXIMIZE_COLS:
        if col in vals.columns:
            vals[col] = -vals[col]
    return vals.to_numpy()


def nondominated_mask_minimize(values):
    """Return mask of non-dominated rows for a minimization problem."""
    values = np.asarray(values, dtype=float)
    n = values.shape[0]
    is_efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_efficient[i]:
            continue
        # Point j dominates i if j is <= i on all objectives and < i on at least one.
        dominates_i = np.all(values <= values[i], axis=1) & np.any(values < values[i], axis=1)
        if np.any(dominates_i):
            is_efficient[i] = False
    return is_efficient

reference_sets = {}
for wf, df in all_results.groupby("welfare_function"):
    df = df.copy().reset_index(drop=True)
    vals = to_minimization_objectives(df)
    mask = nondominated_mask_minimize(vals)
    reference_sets[wf] = df.loc[mask].reset_index(drop=True)

reference_summary = pd.DataFrame([
    {"welfare_function": wf, "n_loaded": len(all_results[all_results["welfare_function"] == wf]), "n_reference": len(ref)}
    for wf, ref in reference_sets.items()
]).sort_values("welfare_function")

display(reference_summary)

for wf, ref in reference_sets.items():
    out = RESULTS_DIR / f"reference_set_{wf.lower()}.csv"
    ref.to_csv(out, index=False)
    print(f"Saved {wf} reference set:", out)


## 6. Select representative policies for the results chapter

The goal is not to declare one universal best policy. Instead, select policies that represent different framings:

- **South Africa justice policy:** good Prioritarian performance, with attention to South Africa's burden after re-evaluation.
- **Global efficiency policy:** good Utilitarian performance.
- **High ambition policy:** lowest `fraction_above_threshold`.
- **Compromise policy:** balanced performance across objectives.

At this stage, selection is based on the Pareto archive only. South Africa-specific burden is added after re-evaluation in Section 8.


In [ ]:
def normalize_minmax(series, minimize=True):
    s = series.astype(float)
    lo, hi = s.min(), s.max()
    if np.isclose(hi, lo):
        return pd.Series(0.0, index=s.index)
    scaled = (s - lo) / (hi - lo)
    return scaled if minimize else 1 - scaled


def add_archive_scores(df):
    df = df.copy()
    # Lower score is better.
    score = 0
    for col in MINIMIZE_COLS:
        score = score + normalize_minmax(df[col], minimize=True)
    for col in MAXIMIZE_COLS:
        score = score + normalize_minmax(df[col], minimize=False)
    df["balanced_archive_score"] = score / len(OBJECTIVE_COLS)
    return df

selected_rows = []

for wf, ref in reference_sets.items():
    ref_scored = add_archive_scores(ref)

    candidates = {
        f"{wf} — best balanced archive score": ref_scored["balanced_archive_score"].idxmin(),
        f"{wf} — lowest climate risk": ref_scored["fraction_above_threshold"].idxmin(),
        f"{wf} — best welfare objective": ref_scored["welfare"].idxmin(),
    }

    for label, idx in candidates.items():
        row = ref_scored.loc[idx].copy()
        row["selection_label"] = label
        selected_rows.append(row)

selected_policies = pd.DataFrame(selected_rows).reset_index(drop=True)

cols_to_show = ["selection_label", "welfare_function", "nfe", "seed"] + OBJECTIVE_COLS + ["balanced_archive_score"]
display(selected_policies[cols_to_show].round(4))

selected_path = RESULTS_DIR / "selected_rival_framing_policies.csv"
selected_policies.to_csv(selected_path, index=False)
print("Saved selected policies:", selected_path)


## 7. Re-evaluate selected policies to compute regional burdens

The Pareto CSV usually stores levers and summary objectives, but not the full regional time series. To calculate South Africa's burden, the selected policies must be re-run in JUSTICE.

Set `RUN_REEVALUATION = True` when you are ready. Start with `N_ENSEMBLES_REEVAL = 1` to test. Increase later for robustness.


In [ ]:
from justice.util.emission_control_constraint import EmissionControlConstraint
from solvers.emodps.rbf import RBF

_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF = 2.0, 0.0


def welfare_from_name(name):
    name = str(name).upper()
    if "PRIOR" in name:
        return WelfareFunction.PRIORITARIAN
    if "UTIL" in name:
        return WelfareFunction.UTILITARIAN
    if "SUFFIC" in name:
        return WelfareFunction.SUFFICIENTARIAN
    if "EGAL" in name:
        return WelfareFunction.EGALITARIAN
    return WelfareFunction.PRIORITARIAN


def run_policy_ecr(policy_row, n_ensemble=1, welfare_function=WelfareFunction.PRIORITARIAN):
    """Reconstruct the RBF policy from one Pareto row and run JUSTICE stepwise."""
    JUSTICE.hard_reset()

    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS_RBF, n_outputs=N_REGIONS)
    c_shape, r_shape, w_shape = rbf.get_shape()

    centers = np.array([policy_row[f"center {i}"] for i in range(c_shape[0])], dtype=float)
    radii = np.array([policy_row[f"radii {i}"] for i in range(r_shape[0])], dtype=float)
    weights = np.array([policy_row[f"weights {i}"] for i in range(w_shape[0])], dtype=float)
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    # Spread ensemble indices across the available range for a small but diverse sample.
    if n_ensemble == 1:
        ensemble_indices = [1]
    else:
        ensemble_indices = list(np.linspace(1, 1000, n_ensemble, dtype=int))

    model = JUSTICE(
        start_year=START_YEAR,
        end_year=END_YEAR,
        timestep=TIMESTEP,
        scenario=REFERENCE_SCENARIO,
        climate_ensembles=ensemble_indices,
        stochastic_run=True,
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function=welfare_function,
    )

    no_ens = model.no_of_ensembles
    ecr = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = 0.0
    diff = 0.0

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )

        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data = model.stepwise_evaluate(timestep=t)
        temp = data["global_temperature"][t, :]

        if t % 5 == 0:
            diff = temp - prev_temp
            prev_temp = temp

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    datasets = model.evaluate()
    datasets["constrained_emission_control_rate"] = constrained_ecr
    return constrained_ecr, datasets, model


In [ ]:
RUN_REEVALUATION = False   # Change to True when you want to re-run selected policies.
N_ENSEMBLES_REEVAL = 1     # Increase later, e.g. 10 or 15.
MAX_POLICIES_TO_RUN = None # Set to an integer for testing, e.g. 2.

REEVAL_DIR = RESULTS_DIR / "rival_framing_reevaluations"
REEVAL_DIR.mkdir(parents=True, exist_ok=True)

if RUN_REEVALUATION:
    rows_to_run = selected_policies if MAX_POLICIES_TO_RUN is None else selected_policies.head(MAX_POLICIES_TO_RUN)

    for i, row in rows_to_run.iterrows():
        label_safe = re.sub(r"[^A-Za-z0-9_]+", "_", row["selection_label"]).strip("_")
        welfare_function = welfare_from_name(row["welfare_function"])

        print(f"Running selected policy {i}: {row['selection_label']} with {welfare_function.name}")
        ecr, datasets, model = run_policy_ecr(
            row,
            n_ensemble=N_ENSEMBLES_REEVAL,
            welfare_function=welfare_function,
        )

        out_path = REEVAL_DIR / f"reeval_{i:02d}_{label_safe}.npy"
        np.save(out_path, {
            "selection_index": int(i),
            "selection_label": row["selection_label"],
            "welfare_function": welfare_function.name,
            "objectives_from_archive": row[OBJECTIVE_COLS].to_dict(),
            "constrained_emission_control_rate": ecr,
            "datasets": datasets,
        })
        print("Saved:", out_path)
else:
    print("RUN_REEVALUATION is False. Skipping re-evaluation.")
    print("Existing re-evaluation files:")
    for p in sorted(REEVAL_DIR.glob("reeval_*.npy")):
        print(" -", p.name)


## 8. South Africa burden comparison

This section loads the re-evaluated policies and computes:

- `abatement_cost / gross_economic_output`
- `economic_damage / gross_economic_output`
- `net_economic_output / gross_economic_output`
- `consumption_per_capita`

The comparison regions include South Africa, peer developing/emerging regions, and high-income regions.


In [ ]:
COMPARE_REGIONS = [
    "zaf",                         # South Africa
    "rsaf", "egy", "noan", "noap", # African/regional comparisons
    "pol", "tur", "mex", "bra", "idn", "chn", "rus", # emerging/fossil-dependent
    "usa", "gbr", "fra", "rfa",   # high-income comparison regions
]

COMPARE_REGIONS = [r for r in COMPARE_REGIONS if r in REGION_LIST]
print("Comparison regions:", COMPARE_REGIONS)


def safe_divide(a, b):
    return np.divide(a, b, out=np.full_like(a, np.nan, dtype=float), where=b != 0)


def summarize_region_for_policy(datasets, ecr, region_code, years=(2050, 2100)):
    idx = REGION_LIST.index(region_code)
    gross = datasets["gross_economic_output"][idx, :, :]
    abat = datasets["abatement_cost"][idx, :, :]
    damage = datasets["economic_damage"][idx, :, :]
    net = datasets["net_economic_output"][idx, :, :]
    cpc = datasets["consumption_per_capita"][idx, :, :]

    abat_burden = safe_divide(abat, gross)
    damage_burden = safe_divide(damage, gross)
    net_share = safe_divide(net, gross)

    row = {
        "region": region_code,
        "mean_abatement_burden": float(np.nanmean(abat_burden)),
        "p95_abatement_burden": float(np.nanpercentile(abat_burden, 95)),
        "max_abatement_burden": float(np.nanmax(abat_burden)),
        "mean_damage_burden": float(np.nanmean(damage_burden)),
        "p95_damage_burden": float(np.nanpercentile(damage_burden, 95)),
        "mean_net_output_share": float(np.nanmean(net_share)),
        "mean_consumption_per_capita": float(np.nanmean(cpc)),
        "mean_ecr": float(np.nanmean(ecr[idx, :, :])),
    }

    for year in years:
        t = _time_horizon.year_to_timestep(year, timestep=TIMESTEP)
        row[f"abatement_burden_{year}"] = float(np.nanmean(abat_burden[t, :]))
        row[f"damage_burden_{year}"] = float(np.nanmean(damage_burden[t, :]))
        row[f"net_output_share_{year}"] = float(np.nanmean(net_share[t, :]))
        row[f"ecr_{year}"] = float(np.nanmean(ecr[idx, t, :]))

    return row


In [ ]:
reeval_files = sorted(REEVAL_DIR.glob("reeval_*.npy"))
print(f"Found {len(reeval_files)} re-evaluation file(s).")

burden_rows = []
policy_rows = []

for file in reeval_files:
    obj = np.load(file, allow_pickle=True).item()
    label = obj["selection_label"]
    welfare_function = obj["welfare_function"]
    datasets = obj["datasets"]
    ecr = obj["constrained_emission_control_rate"]

    policy_summary = {
        "selection_label": label,
        "welfare_function": welfare_function,
        **obj.get("objectives_from_archive", {}),
        "mean_global_temperature_2100": float(np.nanmean(datasets["global_temperature"][TEMP_YEAR_IDX, :])),
        "p95_global_temperature_2100": float(np.nanpercentile(datasets["global_temperature"][TEMP_YEAR_IDX, :], 95)),
    }
    policy_rows.append(policy_summary)

    for region in COMPARE_REGIONS:
        row = summarize_region_for_policy(datasets, ecr, region)
        row["selection_label"] = label
        row["welfare_function"] = welfare_function
        row["source_file"] = str(file)
        burden_rows.append(row)

if burden_rows:
    burden_comparison = pd.DataFrame(burden_rows)
    policy_reeval_summary = pd.DataFrame(policy_rows)

    display(policy_reeval_summary.round(4))
    display(
        burden_comparison
        .sort_values(["selection_label", "mean_abatement_burden"], ascending=[True, False])
        .round(5)
    )

    burden_path = TABLES_DIR / "south_africa_burden_comparison.csv"
    burden_tex = TABLES_DIR / "south_africa_burden_comparison.tex"
    policy_path = TABLES_DIR / "policy_reevaluation_summary.csv"

    burden_comparison.to_csv(burden_path, index=False)
    policy_reeval_summary.to_csv(policy_path, index=False)

    # Compact LaTeX table: only South Africa + selected developed comparison regions.
    compact_regions = ["zaf", "usa", "gbr", "fra", "rfa"]
    compact = burden_comparison[burden_comparison["region"].isin(compact_regions)].copy()
    compact_cols = [
        "selection_label", "welfare_function", "region",
        "mean_abatement_burden", "abatement_burden_2100",
        "mean_damage_burden", "mean_ecr",
    ]
    compact[compact_cols].to_latex(
        burden_tex,
        index=False,
        float_format="%.4f",
        caption="Relative burden comparison for South Africa and selected high-income regions",
        label="tab:zaf_burden_comparison",
        escape=True,
    )

    print("Saved:", burden_path)
    print("Saved:", policy_path)
    print("Saved:", burden_tex)
else:
    print("No re-evaluation files loaded yet. Run Section 7 first.")


## 9. Rival welfare comparison

This is the key comparison for the rival framing section:

- **Prioritarian:** gives more weight to worse-off regions.
- **Utilitarian:** focuses on aggregate global performance.

The question is whether Utilitarian/global efficiency policies perform well globally while leaving South Africa with a higher relative burden.


In [ ]:
if "burden_comparison" in globals() and not burden_comparison.empty:
    zaf_only = burden_comparison[burden_comparison["region"] == "zaf"].copy()

    rival_welfare_table = zaf_only[[
        "selection_label", "welfare_function",
        "mean_abatement_burden", "p95_abatement_burden", "abatement_burden_2100",
        "mean_damage_burden", "damage_burden_2100",
        "mean_net_output_share", "mean_consumption_per_capita", "mean_ecr",
    ]].sort_values(["welfare_function", "mean_abatement_burden"])

    display(rival_welfare_table.round(5))

    rival_csv = TABLES_DIR / "rival_welfare_comparison_zaf.csv"
    rival_tex = TABLES_DIR / "rival_welfare_comparison_zaf.tex"
    rival_welfare_table.to_csv(rival_csv, index=False)
    rival_welfare_table.to_latex(
        rival_tex,
        index=False,
        float_format="%.4f",
        caption="Rival welfare comparison for South Africa-specific burden indicators",
        label="tab:rival_welfare_comparison_zaf",
        escape=True,
    )

    print("Saved:", rival_csv)
    print("Saved:", rival_tex)
else:
    print("burden_comparison not available yet. Run Section 8 after re-evaluation.")


## 10. Robustness outcomes

There are two levels of robustness you can use:

1. **Archive-level robustness:** uses Pareto summary objectives across loaded policies.
2. **Re-evaluation robustness:** uses the re-run policies and multiple ensembles/scenarios if available.

This notebook provides a simple results table for the re-evaluated policies. If you later re-evaluate under more scenarios, add a `scenario` column to the saved outputs and group by policy.


In [ ]:
if "policy_reeval_summary" in globals() and not policy_reeval_summary.empty:
    robustness_table = policy_reeval_summary.copy()

    # If fraction_above_threshold was saved in the archive, use it directly.
    # Otherwise temperature in 2100 is still useful as a climate-risk proxy.
    show_cols = [c for c in [
        "selection_label", "welfare_function", "welfare", "fraction_above_threshold",
        "welfare_loss_damage", "welfare_loss_abatement",
        "mean_global_temperature_2100", "p95_global_temperature_2100",
    ] if c in robustness_table.columns]

    display(robustness_table[show_cols].round(4))

    robustness_csv = TABLES_DIR / "robustness_outcomes_selected_policies.csv"
    robustness_tex = TABLES_DIR / "robustness_outcomes_selected_policies.tex"
    robustness_table[show_cols].to_csv(robustness_csv, index=False)
    robustness_table[show_cols].to_latex(
        robustness_tex,
        index=False,
        float_format="%.4f",
        caption="Robustness outcomes for selected rival-framing policies",
        label="tab:robustness_selected_policies",
        escape=True,
    )

    print("Saved:", robustness_csv)
    print("Saved:", robustness_tex)
else:
    print("policy_reeval_summary not available yet. Run Section 8 after re-evaluation.")


## 11. Results chapter tables

This section creates the four tables you said you want in Chapter 4:

1. What policies perform well.
2. South Africa burden comparison.
3. Robustness outcomes.
4. Rival welfare comparison.

The tables are saved as `.csv` and `.tex` in `A- Project G15/tables/`.


In [ ]:
# 1. What policies perform well: from archive selection
what_performs_well = selected_policies[cols_to_show].copy()
what_performs_well_csv = TABLES_DIR / "chapter4_what_policies_perform_well.csv"
what_performs_well_tex = TABLES_DIR / "chapter4_what_policies_perform_well.tex"

what_performs_well.to_csv(what_performs_well_csv, index=False)
what_performs_well.to_latex(
    what_performs_well_tex,
    index=False,
    float_format="%.4f",
    caption="Selected policies representing different model framings",
    label="tab:what_policies_perform_well",
    escape=True,
)

print("Saved:", what_performs_well_csv)
print("Saved:", what_performs_well_tex)

# 2, 3, 4 are saved in Sections 8, 9, and 10 after re-evaluation.
print("\nExpected Chapter 4 table files:")
for p in sorted(TABLES_DIR.glob("*.tex")):
    print(" -", p.name)


## 12. Debate translation

Use these model-based claims in the debate.


In [ ]:
debate_claims = pd.DataFrame([
    {
        "Rival position": "Utilitarian / global efficiency",
        "Model evidence to use": "Compare Utilitarian and Prioritarian selected policies; show zaf abatement burden.",
        "Debate line": "Aggregate global welfare hides who pays for the transition."
    },
    {
        "Rival position": "Equal obligations",
        "Model evidence to use": "Compare emission_control_rate and abatement_cost/gross_output for zaf vs usa/gbr/fra/rfa.",
        "Debate line": "Equal mitigation rates do not create equal burdens."
    },
    {
        "Rival position": "High ambition / survival",
        "Model evidence to use": "Show low fraction_above_threshold policies and their South Africa burden.",
        "Debate line": "We share the urgency, but ambition without finance is not implementable."
    },
    {
        "Rival position": "Loan-heavy finance package",
        "Model evidence to use": "Use zaf abatement_cost as the transition burden that finance must cover.",
        "Debate line": "Loans are not the same as support if they shift transition costs into South Africa's future debt."
    },
])

display(debate_claims)

debate_claims.to_csv(TABLES_DIR / "debate_claims_from_model.csv", index=False)
debate_claims.to_latex(
    TABLES_DIR / "debate_claims_from_model.tex",
    index=False,
    caption="Debate claims derived from rival framing model results",
    label="tab:debate_claims_from_model",
    escape=True,
)


## 13. Short text for Chapter 4

You can paste this text above the results tables and edit it once the numbers are available.


In [ ]:
chapter4_text = r"""
The results are interpreted through four rival framings. The South Africa justice framing uses a Prioritarian welfare lens and evaluates whether mitigation costs are proportional to South Africa's economic capacity. The global efficiency framing uses a Utilitarian welfare lens and focuses on aggregate model performance. The equal obligation framing compares mitigation effort across regions, while the high-ambition framing prioritises temperature and threshold outcomes.

This distinction is important because the same Pareto set can support different political conclusions. A policy that performs well on aggregate global welfare may still be unacceptable for South Africa if it imposes a high abatement burden relative to gross economic output. Therefore, the analysis separates technical model performance from political acceptability under South Africa's mandate.
"""

print(chapter4_text)
